# ConnectX Kaggle Submission vs Reference Lookahead Tuner

This notebook imports:

- `main_corrected.py` / `main.py` as the Kaggle submission agent
- `fast_connect4_lookahead.py` as the standalone reference lookahead agent

It then runs local head-to-head matches so you can tune:

- submission LA switch stone count
- submission late lookahead depth
- PPO mirror TTA
- PPO-guided LA root bias
- book randomization
- reference lookahead depth
- reference lookahead heuristic knobs

Put this notebook in the same folder as:

```text
main_corrected.py
fast_connect4_lookahead.py
PPO_2004.pt
```

First run may be slow because the Numba lookahead compiles. The first move always wants to be special. Diva behavior, but useful.


In [1]:
# ============================================================
# User configuration
# ============================================================

from pathlib import Path

BASE_DIR = Path.cwd()
#SUBMISSION_PATH = BASE_DIR / "main_old.py"
SUBMISSION_PATH = BASE_DIR / "main.py"
LOOKAHEAD_PATH = BASE_DIR / "fast_connect4_lookahead.py"
MODEL_FILE = BASE_DIR / "PPO_2004.pt"

# Match settings
GAMES_PER_SIDE = 5          # total games = 2 * GAMES_PER_SIDE
RANDOM_SEED = 666
MAX_PLIES = 42
TORCH_THREADS = 1

# Warn if a move takes longer than this locally. This does not interrupt the move.
MOVE_TIME_WARNING_SEC = 1.90

print("Submission:", SUBMISSION_PATH)
print("Reference LA:", LOOKAHEAD_PATH)
print("Model:", MODEL_FILE, "exists:", MODEL_FILE.exists())


Submission: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\main.py
Reference LA: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\fast_connect4_lookahead.py
Model: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\PPO_2004.pt exists: True


In [2]:
# ============================================================
# Imports and module loader
# ============================================================

import os
import sys
import time
import random
import importlib.util
from types import SimpleNamespace
from dataclasses import dataclass, asdict
from typing import Callable, Dict, List, Optional, Any

import numpy as np
import pandas as pd

try:
    import torch
    torch.set_num_threads(TORCH_THREADS)
except Exception as e:
    print("Torch thread setup skipped:", repr(e))

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

ROWS, COLS, INAROW = 6, 7, 4
CENTER_ORDER = (3, 4, 2, 5, 1, 6, 0)

def import_module_from_path(module_name: str, path: Path, reload: bool = True):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    if reload and module_name in sys.modules:
        del sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

submission = import_module_from_path("kaggle_submission_under_test", SUBMISSION_PATH)
la_mod = import_module_from_path("reference_fast_connect4_lookahead", LOOKAHEAD_PATH)

print("Imported submission module:", submission.__name__)
print("Imported reference LA module:", la_mod.__name__)
print("Has submission.agent:", hasattr(submission, "agent"))
print("Has Connect4Lookahead:", hasattr(la_mod, "Connect4Lookahead"))


Imported submission module: kaggle_submission_under_test
Imported reference LA module: reference_fast_connect4_lookahead
Has submission.agent: True
Has Connect4Lookahead: True


In [3]:
# ============================================================
# Knobs: edit this cell most often
# ============================================================

SUBMISSION_KNOBS = {
    # Total stones on board BEFORE the submission move.
    # 20 = your current Kaggle sweet spot.
    "LA_TAKES_OVER_AT_STONES": 20,

    # Embedded pure-Python Kaggle lookahead depth.
    # This is not equivalent to Numba reference LA13, because it is time-budgeted.
    "LATE_LOOKAHEAD_STEPS": 13,

    # For clean deterministic benchmarking, keep False.
    "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,

    # PPO inference uses normal + mirrored board logits.
    "PPO_MIRROR_TTA": False,

    # PPO guides late lookahead root move ordering / scoring.
    "PPO_GUIDED_LA_ROOT": False,

    # Start conservative. 80 may be too strong.
    "PPO_LA_ROOT_BIAS": 0.0,
}

REFERENCE_LA_DEPTH = 13 #13

REFERENCE_LA_KNOBS = {
    # Useful high-level toggles
    "OPENING_BOOK": True,
    "OPENING_RANDOM": False,
    "DEPTH_BASED_FLOATING": True,
    "DOUBLE_THREAT_GUARD": True,
    "FORK_REPLY_GUARD": True,

    # Heuristic knobs
    "DEFENSIVE": 1.55,
    "FLOATING_NEAR": 0.25,
    "FLOATING_FAR": 0.125,
    "CENTER_BONUS": 4.0,
    "PARITY_BONUS": 0.75,
    "VERT_MUL": 0.8,
    "TEMPO_W": 72.5,
    "PARITY_MOVE_W": 1.75,
    "PARITY_UNLOCK_W": 0.25,
    "THREATSPACE_W": 9.0,
}

print("Submission knobs:")
display(pd.Series(SUBMISSION_KNOBS).to_frame("value"))

print("Reference LA knobs:")
display(pd.Series({"depth": REFERENCE_LA_DEPTH, **REFERENCE_LA_KNOBS}).to_frame("value"))

Submission knobs:


,value
LA_TAKES_OVER_AT_STONES,20
LATE_LOOKAHEAD_STEPS,13
RANDOMIZE_SECOND_PLAYER_BOOK_REPLY,False
PPO_MIRROR_TTA,False
PPO_GUIDED_LA_ROOT,False
PPO_LA_ROOT_BIAS,0.0


Reference LA knobs:


,value
depth,13
OPENING_BOOK,True
OPENING_RANDOM,False
DEPTH_BASED_FLOATING,True
DOUBLE_THREAT_GUARD,True
FORK_REPLY_GUARD,True
DEFENSIVE,1.55
FLOATING_NEAR,0.25
FLOATING_FAR,0.125
CENTER_BONUS,4.0


In [4]:
# ============================================================
# ConnectX board utilities
# Board convention: top-row-first numpy array, values 0/1/2
# ============================================================

def empty_board() -> np.ndarray:
    return np.zeros((ROWS, COLS), dtype=np.int8)

def legal_moves(board: np.ndarray) -> List[int]:
    return [c for c in range(COLS) if int(board[0, c]) == 0]

def drop_piece(board: np.ndarray, col: int, mark: int) -> Optional[int]:
    if col < 0 or col >= COLS or board[0, col] != 0:
        return None
    for r in range(ROWS - 1, -1, -1):
        if board[r, col] == 0:
            board[r, col] = mark
            return r
    return None

def has_four_from(board: np.ndarray, row: int, col: int, mark: int) -> bool:
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
        count = 1

        rr, cc = row + dr, col + dc
        while 0 <= rr < ROWS and 0 <= cc < COLS and board[rr, cc] == mark:
            count += 1
            rr += dr
            cc += dc

        rr, cc = row - dr, col - dc
        while 0 <= rr < ROWS and 0 <= cc < COLS and board[rr, cc] == mark:
            count += 1
            rr -= dr
            cc -= dc

        if count >= INAROW:
            return True
    return False

def check_winner_full(board: np.ndarray) -> int:
    for r in range(ROWS):
        for c in range(COLS):
            mark = int(board[r, c])
            if mark and has_four_from(board, r, c, mark):
                return mark
    return 0

def board_to_obs(board: np.ndarray, mark: int) -> Dict[str, Any]:
    return {
        "board": board.astype(int).reshape(-1).tolist(),
        "mark": int(mark),
    }

KAGGLE_CONFIG = SimpleNamespace(
    rows=ROWS,
    columns=COLS,
    inarow=INAROW,
    timeout=2,
    actTimeout=2,
)

def render_board(board: np.ndarray) -> str:
    chars = {0: ".", 1: "X", 2: "O"}
    lines = [" ".join(chars[int(v)] for v in row) for row in board]
    return "\n".join(lines) + "\n0 1 2 3 4 5 6"

def show_board(board: np.ndarray):
    print(render_board(board))


In [5]:
# ============================================================
# Agent wrappers and configuration helpers
# ============================================================

def configure_submission_module(module, knobs: Dict[str, Any], model_file: Path):
    # Set local model path. Absolute path makes local notebook execution robust.
    if hasattr(module, "MODEL_FILE"):
        module.MODEL_FILE = str(Path(model_file).resolve())

    for k, v in knobs.items():
        setattr(module, k, v)

    # Keep already-loaded model unless the path was changed manually.
    # To force reload:
    # submission._MODEL = None
    return module

def make_submission_agent(module, knobs: Dict[str, Any], model_file: Path, name: str = "SUB"):
    configure_submission_module(module, knobs, model_file)

    def _agent(board: np.ndarray, mark: int) -> int:
        obs = board_to_obs(board, mark)
        return int(module.agent(obs, KAGGLE_CONFIG))

    _agent.name = name
    _agent.kind = "submission"
    _agent.knobs = dict(knobs)
    return _agent

def make_reference_la_agent(la_module, depth: int, knobs: Dict[str, Any], name: Optional[str] = None):
    engine = la_module.Connect4Lookahead()

    for k, v in knobs.items():
        setattr(engine, k, v)

    agent_name = name or f"LA{depth}"

    def _agent(board: np.ndarray, mark: int) -> int:
        return int(engine.n_step_lookahead(board, mark, depth=int(depth)))

    _agent.name = agent_name
    _agent.kind = "reference_lookahead"
    _agent.depth = int(depth)
    _agent.knobs = dict(knobs)
    _agent.engine = engine
    return _agent

def reset_submission_model_cache(module):
    if hasattr(module, "_MODEL"):
        module._MODEL = None

def warmup_submission(module, board: Optional[np.ndarray] = None):
    """Loads the PPO model once. Useful before timed local sweeps."""
    if board is None:
        board = empty_board()
    if hasattr(module, "_load_model_once"):
        t0 = time.perf_counter()
        module._load_model_once()
        print(f"Submission model loaded in {time.perf_counter() - t0:.3f}s")
    else:
        print("No _load_model_once() found.")


In [6]:
# ============================================================
# Game runner
# ============================================================

@dataclass
class GameResult:
    winner: int                 # 0 draw, 1 player one, 2 player two
    submission_mark: int        # 1 or 2
    result_for_submission: str  # win/loss/draw/error
    plies: int
    moves: List[int]
    board: np.ndarray
    error: Optional[str]
    invalid_by: Optional[int]
    max_move_sec: float
    avg_submission_sec: float
    avg_reference_sec: float
    move_log: List[Dict[str, Any]]

def play_game(
    player1: Callable[[np.ndarray, int], int],
    player2: Callable[[np.ndarray, int], int],
    submission_mark: int,
    seed: Optional[int] = None,
    max_plies: int = MAX_PLIES,
    verbose: bool = False,
) -> GameResult:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    board = empty_board()
    moves = []
    move_log = []
    agents = {1: player1, 2: player2}
    winner = 0
    error = None
    invalid_by = None

    for ply in range(max_plies):
        mark = 1 if (ply % 2 == 0) else 2
        agent_fn = agents[mark]
        agent_name = getattr(agent_fn, "name", f"P{mark}")

        legal = legal_moves(board)
        if not legal:
            break

        board_before = board.copy()
        t0 = time.perf_counter()
        try:
            move = int(agent_fn(board.copy(), mark))
        except Exception as e:
            dt = time.perf_counter() - t0
            error = f"{agent_name} exception: {type(e).__name__}: {e}"
            invalid_by = mark
            winner = 3 - mark
            move_log.append({
                "ply": ply,
                "mark": mark,
                "agent": agent_name,
                "move": None,
                "legal": legal,
                "sec": dt,
                "error": error,
            })
            break

        dt = time.perf_counter() - t0

        if move not in legal:
            error = f"{agent_name} invalid move {move}; legal={legal}"
            invalid_by = mark
            winner = 3 - mark
            move_log.append({
                "ply": ply,
                "mark": mark,
                "agent": agent_name,
                "move": move,
                "legal": legal,
                "sec": dt,
                "error": error,
            })
            break

        row = drop_piece(board, move, mark)
        moves.append(move)
        move_log.append({
            "ply": ply,
            "mark": mark,
            "agent": agent_name,
            "move": move,
            "legal": legal,
            "sec": dt,
            "warning_slow": dt >= MOVE_TIME_WARNING_SEC,
            "board_before": board_before,
        })

        if verbose:
            print(f"ply {ply:02d} P{mark} {agent_name} -> col {move} ({dt:.3f}s)")
            show_board(board)
            print()

        if row is not None and has_four_from(board, row, move, mark):
            winner = mark
            break

    if winner == 0 and len(legal_moves(board)) == 0:
        result_for_submission = "draw"
    elif error is not None:
        result_for_submission = "error" if invalid_by == submission_mark else "win"
    elif winner == 0:
        result_for_submission = "draw"
    elif winner == submission_mark:
        result_for_submission = "win"
    else:
        result_for_submission = "loss"

    sub_times = [m["sec"] for m in move_log if m.get("agent") == "SUB"]
    ref_times = [m["sec"] for m in move_log if m.get("agent", "").startswith("LA")]

    return GameResult(
        winner=int(winner),
        submission_mark=int(submission_mark),
        result_for_submission=result_for_submission,
        plies=len(moves),
        moves=moves,
        board=board.copy(),
        error=error,
        invalid_by=invalid_by,
        max_move_sec=max([m["sec"] for m in move_log], default=0.0),
        avg_submission_sec=float(np.mean(sub_times)) if sub_times else 0.0,
        avg_reference_sec=float(np.mean(ref_times)) if ref_times else 0.0,
        move_log=move_log,
    )

def summarize_results(results: List[GameResult]) -> pd.DataFrame:
    if not results:
        return pd.DataFrame()

    n = len(results)
    wins = sum(r.result_for_submission == "win" for r in results)
    losses = sum(r.result_for_submission == "loss" for r in results)
    draws = sum(r.result_for_submission == "draw" for r in results)
    errors = sum(r.error is not None for r in results)

    as_first = [r for r in results if r.submission_mark == 1]
    as_second = [r for r in results if r.submission_mark == 2]

    def rate(rows, label):
        if not rows:
            return {
                f"{label}_games": 0,
                f"{label}_wins": 0,
                f"{label}_losses": 0,
                f"{label}_draws": 0,
                f"{label}_score_rate": np.nan,
            }
        w = sum(r.result_for_submission == "win" for r in rows)
        l = sum(r.result_for_submission == "loss" for r in rows)
        d = sum(r.result_for_submission == "draw" for r in rows)
        return {
            f"{label}_games": len(rows),
            f"{label}_wins": w,
            f"{label}_losses": l,
            f"{label}_draws": d,
            f"{label}_score_rate": (w + 0.5 * d) / len(rows),
        }

    row = {
        "games": n,
        "wins": wins,
        "losses": losses,
        "draws": draws,
        "score_rate": (wins + 0.5 * draws) / n,
        "win_rate": wins / n,
        "loss_rate": losses / n,
        "draw_rate": draws / n,
        "errors": errors,
        "avg_plies": float(np.mean([r.plies for r in results])),
        "max_move_sec": float(max(r.max_move_sec for r in results)),
        "avg_submission_sec": float(np.mean([r.avg_submission_sec for r in results])),
        "avg_reference_sec": float(np.mean([r.avg_reference_sec for r in results])),
    }
    row.update(rate(as_first, "as_first"))
    row.update(rate(as_second, "as_second"))

    return pd.DataFrame([row])

def results_to_games_df(results: List[GameResult]) -> pd.DataFrame:
    rows = []
    for i, r in enumerate(results):
        rows.append({
            "game": i,
            "submission_mark": r.submission_mark,
            "winner": r.winner,
            "result_for_submission": r.result_for_submission,
            "plies": r.plies,
            "moves": " ".join(map(str, r.moves)),
            "error": r.error,
            "max_move_sec": r.max_move_sec,
            "avg_submission_sec": r.avg_submission_sec,
            "avg_reference_sec": r.avg_reference_sec,
        })
    return pd.DataFrame(rows)

def run_match(
    submission_knobs: Dict[str, Any],
    reference_depth: int,
    reference_knobs: Dict[str, Any],
    games_per_side: int = GAMES_PER_SIDE,
    seed: int = RANDOM_SEED,
    verbose_first_game: bool = False,
) -> List[GameResult]:
    sub_agent = make_submission_agent(submission, submission_knobs, MODEL_FILE, name="SUB")
    la_agent = make_reference_la_agent(la_mod, reference_depth, reference_knobs, name=f"LA{reference_depth}")

    results = []

    # SUB as first player.
    for i in tqdm(range(games_per_side), desc=f"SUB first vs LA{reference_depth}"):
        results.append(play_game(
            player1=sub_agent,
            player2=la_agent,
            submission_mark=1,
            seed=seed + i,
            verbose=(verbose_first_game and i == 0),
        ))

    # SUB as second player.
    for i in tqdm(range(games_per_side), desc=f"SUB second vs LA{reference_depth}"):
        results.append(play_game(
            player1=la_agent,
            player2=sub_agent,
            submission_mark=2,
            seed=seed + 10000 + i,
            verbose=False,
        ))

    return results


In [7]:
# ============================================================
# Optional warmup
# ============================================================

# This will fail if PPO_2004.pt is not next to the notebook.
# That is useful: better to fail here than on game 17 like a gremlin.
#
# Uncomment when your checkpoint is in place:
#
configure_submission_module(submission, SUBMISSION_KNOBS, MODEL_FILE)
warmup_submission(submission)


Submission model loaded in 0.011s


C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\main.py:943: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_

In [8]:
# ============================================================
# Run one configured matchup
# ============================================================

results = run_match(
    submission_knobs=SUBMISSION_KNOBS,
    reference_depth=REFERENCE_LA_DEPTH,
    reference_knobs=REFERENCE_LA_KNOBS,
    games_per_side=GAMES_PER_SIDE,
    seed=RANDOM_SEED,
    verbose_first_game=False,
)

summary_df = summarize_results(results)
games_df = results_to_games_df(results)

display(summary_df)
display(games_df)


SUB first vs LA13:   0%|          | 0/5 [00:00<?, ?it/s]

SUB second vs LA13:   0%|          | 0/5 [00:00<?, ?it/s]

,games,wins,losses,draws,score_rate,win_rate,loss_rate,draw_rate,errors,avg_plies,...,as_first_games,as_first_wins,as_first_losses,as_first_draws,as_first_score_rate,as_second_games,as_second_wins,as_second_losses,as_second_draws,as_second_score_rate
0,10,0,10,0,0.0,0.0,1.0,0.0,0,39.5,...,5,0,5,0,0.0,5,0,5,0,0.0


,game,submission_mark,winner,result_for_submission,plies,moves,error,max_move_sec,avg_submission_sec,avg_reference_sec
0,0,1,2,loss,40,3 2 5 3 5 2 3 3 2 2 3 5 5 2 4 6 4 4 4 3 5 4 6 ...,None,10.756784,0.305196,1.214897
1,1,1,2,loss,40,3 2 5 3 5 2 3 3 2 2 3 5 5 2 4 6 4 4 4 3 5 4 6 ...,None,4.780951,0.354806,0.910008
2,2,1,2,loss,40,3 2 5 3 5 2 3 3 2 2 3 5 5 2 4 6 4 4 4 3 5 4 6 ...,None,4.862015,0.319316,0.930662
3,3,1,2,loss,40,3 2 5 3 5 2 3 3 2 2 3 5 5 2 4 6 4 4 4 3 5 4 6 ...,None,5.820763,0.354073,1.020850
4,4,1,2,loss,40,3 2 5 3 5 2 3 3 2 2 3 5 5 2 4 6 4 4 4 3 5 4 6 ...,None,5.573856,0.351556,1.000793
5,5,2,1,loss,39,3 2 5 3 3 3 6 4 4 4 4 1 3 1 2 4 2 2 2 1 1 3 2 ...,None,4.494076,0.349079,0.538003
6,6,2,1,loss,39,3 2 5 3 3 3 6 4 4 4 4 1 3 1 2 4 2 2 2 1 1 3 2 ...,None,4.459178,0.294315,0.530121
7,7,2,1,loss,39,3 2 5 3 3 3 6 4 4 4 4 1 3 1 2 4 2 2 2 1 1 3 2 ...,None,4.605097,0.347049,0.557614
8,8,2,1,loss,39,3 2 5 3 3 3 6 4 4 4 4 1 3 1 2 4 2 2 2 1 1 3 2 ...,None,4.567447,0.343360,0.531686
9,9,2,1,loss,39,3 2 5 3 3 3 6 4 4 4 4 1 3 1 2 4 2 2 2 1 1 3 2 ...,None,4.549814,0.348868,0.525653


In [9]:
# ============================================================
# Inspect a game
# ============================================================

GAME_INDEX = 0

r = results[GAME_INDEX]
print(f"Game {GAME_INDEX}")
print("submission_mark:", r.submission_mark)
print("winner:", r.winner)
print("result_for_submission:", r.result_for_submission)
print("plies:", r.plies)
print("moves:", r.moves)
print("error:", r.error)
print()
show_board(r.board)

move_log_df = pd.DataFrame([
    {
        "ply": m["ply"],
        "mark": m["mark"],
        "agent": m["agent"],
        "move": m["move"],
        "sec": m["sec"],
        "warning_slow": m.get("warning_slow", False),
        "error": m.get("error"),
    }
    for m in r.move_log
])
display(move_log_df)


Game 0
submission_mark: 1
winner: 2
result_for_submission: loss
plies: 40
moves: [3, 2, 5, 3, 5, 2, 3, 3, 2, 2, 3, 5, 5, 2, 4, 6, 4, 4, 4, 3, 5, 4, 6, 4, 0, 5, 2, 0, 6, 6, 0, 1, 1, 1, 0, 1, 0, 0, 6, 6]
error: None

O . X O O O O
X . O X O X X
X O O O X X O
X O X X O O X
O X O O X X X
X O O X X X O
0 1 2 3 4 5 6


,ply,mark,agent,move,sec,warning_slow,error
0,0,1,SUB,3,0.002343,False,None
1,1,2,LA13,2,0.000015,False,None
2,2,1,SUB,5,0.002301,False,None
3,3,2,LA13,3,10.756784,True,None
4,4,1,SUB,5,0.006293,False,None
5,5,2,LA13,2,4.811072,True,None
6,6,1,SUB,3,0.005952,False,None
7,7,2,LA13,3,1.916769,True,None
8,8,1,SUB,2,0.005271,False,None
9,9,2,LA13,2,1.374291,False,None


In [10]:
# ============================================================
# Sweep template: find first reference LA depth we can beat
# ============================================================

SUBMISSION_SWEEP = [

    {
        "label": "S18_LA11",
        **SUBMISSION_KNOBS,
        "PPO_MIRROR_TTA": False,
        "PPO_GUIDED_LA_ROOT": False,
        "PPO_LA_ROOT_BIAS": 0.0,
        "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,
        "LA_TAKES_OVER_AT_STONES": 18,
        "LATE_LOOKAHEAD_STEPS": 11,
    },
    {
        "label": "S20_LA11",
        **SUBMISSION_KNOBS,
        "PPO_MIRROR_TTA": False,
        "PPO_GUIDED_LA_ROOT": False,
        "PPO_LA_ROOT_BIAS": 0.0,
        "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,
        "LA_TAKES_OVER_AT_STONES": 20,
        "LATE_LOOKAHEAD_STEPS": 11,
    },
    {
        "label": "S22_LA11",
        **SUBMISSION_KNOBS,
        "PPO_MIRROR_TTA": False,
        "PPO_GUIDED_LA_ROOT": False,
        "PPO_LA_ROOT_BIAS": 0.0,
        "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,
        "LA_TAKES_OVER_AT_STONES": 22,
        "LATE_LOOKAHEAD_STEPS": 11,
    },
    {
        "label": "S20_LA12",
        **SUBMISSION_KNOBS,
        "PPO_MIRROR_TTA": False,
        "PPO_GUIDED_LA_ROOT": False,
        "PPO_LA_ROOT_BIAS": 0.0,
        "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,
        "LA_TAKES_OVER_AT_STONES": 20,
        "LATE_LOOKAHEAD_STEPS": 12,
    },
    {
        "label": "S22_LA12",
        **SUBMISSION_KNOBS,
        "PPO_MIRROR_TTA": False,
        "PPO_GUIDED_LA_ROOT": False,
        "PPO_LA_ROOT_BIAS": 0.0,
        "RANDOMIZE_SECOND_PLAYER_BOOK_REPLY": False,
        "LA_TAKES_OVER_AT_STONES": 22,
        "LATE_LOOKAHEAD_STEPS": 12,
    },
]

# First sweep: one deterministic game per side at each depth.
# Since both agents are mostly deterministic, this is usually enough to find the breakpoint.
REFERENCE_DEPTH_SWEEP = [10, 11, 12, 13]
SWEEP_GAMES_PER_SIDE = 3


def run_sweep(
    submission_sweep: List[Dict[str, Any]],
    reference_depths: List[int],
    reference_knobs: Dict[str, Any],
    games_per_side: int = SWEEP_GAMES_PER_SIDE,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    rows = []

    for cfg_i, cfg in enumerate(submission_sweep):
        label = cfg.get("label", f"cfg_{cfg_i}")
        knobs = {k: v for k, v in cfg.items() if k != "label"}

        for depth in reference_depths:
            print(f"\n=== {label} vs reference LA{depth} ===")

            res = run_match(
                submission_knobs=knobs,
                reference_depth=depth,
                reference_knobs=reference_knobs,
                games_per_side=games_per_side,
                seed=seed + cfg_i * 100000 + depth * 1000,
                verbose_first_game=False,
            )

            s = summarize_results(res).iloc[0].to_dict()
            s.update({
                "label": label,
                "reference_depth": depth,
                **{f"sub_{k}": v for k, v in knobs.items()},
            })
            rows.append(s)

            display(pd.DataFrame([s])[[
                "label",
                "reference_depth",
                "games",
                "wins",
                "losses",
                "draws",
                "score_rate",
                "as_first_score_rate",
                "as_second_score_rate",
                "avg_plies",
                "max_move_sec",
            ]])

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            ["reference_depth", "score_rate", "as_second_score_rate"],
            ascending=[True, False, False],
        ).reset_index(drop=True)

    return df


sweep_df = run_sweep(
    SUBMISSION_SWEEP,
    REFERENCE_DEPTH_SWEEP,
    REFERENCE_LA_KNOBS,
    games_per_side=SWEEP_GAMES_PER_SIDE,
)

display(sweep_df[[
    "label",
    "reference_depth",
    "games",
    "wins",
    "losses",
    "draws",
    "score_rate",
    "as_first_score_rate",
    "as_second_score_rate",
    "avg_plies",
    "max_move_sec",
]])


=== S18_LA11 vs reference LA10 ===


SUB first vs LA10:   0%|          | 0/3 [00:00<?, ?it/s]

SUB second vs LA10:   0%|          | 0/3 [00:00<?, ?it/s]

,label,reference_depth,games,wins,losses,draws,score_rate,as_first_score_rate,as_second_score_rate,avg_plies,max_move_sec
0,S18_LA11,10,6.0,6.0,0.0,0.0,1.0,1.0,1.0,32.833333,2.902492



=== S18_LA11 vs reference LA11 ===


...


=== S22_LA12 vs reference LA13 ===


SUB first vs LA13:   0%|          | 0/3 [00:00<?, ?it/s]

SUB second vs LA13:   0%|          | 0/3 [00:00<?, ?it/s]

,label,reference_depth,games,wins,losses,draws,score_rate,as_first_score_rate,as_second_score_rate,avg_plies,max_move_sec
0,S22_LA12,13,6.0,0.0,6.0,0.0,0.0,0.0,0.0,40.5,5.529481


,label,reference_depth,games,wins,losses,draws,score_rate,as_first_score_rate,as_second_score_rate,avg_plies,max_move_sec
0,S18_LA11,10,6.0,6.0,0.0,0.0,1.000000,1.0,1.000000,32.833333,2.902492
1,S20_LA11,10,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,36.000000,0.341635
2,S22_LA11,10,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,33.000000,0.368632
3,S20_LA12,10,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,36.000000,0.387461
4,S22_LA12,10,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,33.000000,0.383504
5,S18_LA11,11,6.0,6.0,0.0,0.0,1.000000,1.0,1.000000,40.500000,1.593764
6,S20_LA11,11,6.0,6.0,0.0,0.0,1.000000,1.0,1.000000,40.500000,1.098515
7,S20_LA12,11,6.0,6.0,0.0,0.0,1.000000,1.0,1.000000,39.500000,1.227248
8,S22_LA11,11,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,36.000000,0.890307
9,S22_LA12,11,6.0,3.0,3.0,0.0,0.500000,1.0,0.000000,35.000000,0.941051


In [11]:
# ============================================================
# Breakpoint summary
# ============================================================

def breakpoint_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for label, g in df.groupby("label"):
        g = g.sort_values("reference_depth")

        beat_depths = g.loc[g["score_rate"] > 0.5, "reference_depth"].tolist()
        draw_or_better = g.loc[g["score_rate"] >= 0.5, "reference_depth"].tolist()
        nonzero = g.loc[g["score_rate"] > 0.0, "reference_depth"].tolist()

        best_idx = g["score_rate"].idxmax()

        rows.append({
            "label": label,
            "best_score_rate": g.loc[best_idx, "score_rate"],
            "best_depth": int(g.loc[best_idx, "reference_depth"]),

            "max_beat_depth": max(beat_depths) if beat_depths else None,
            "max_draw_or_better_depth": max(draw_or_better) if draw_or_better else None,
            "max_nonzero_depth": max(nonzero) if nonzero else None,

            "avg_score_rate": g["score_rate"].mean(),
            "avg_as_first_score_rate": g["as_first_score_rate"].mean(),
            "avg_as_second_score_rate": g["as_second_score_rate"].mean(),

            "avg_max_move_sec": g["max_move_sec"].mean(),
            "worst_max_move_sec": g["max_move_sec"].max(),
        })

    return pd.DataFrame(rows).sort_values(
        [
            "max_draw_or_better_depth",
            "max_nonzero_depth",
            "best_score_rate",
            "avg_score_rate",
        ],
        ascending=False,
    ).reset_index(drop=True)


bp_df = breakpoint_summary(sweep_df)
display(bp_df)

,label,best_score_rate,best_depth,max_beat_depth,max_draw_or_better_depth,max_nonzero_depth,avg_score_rate,avg_as_first_score_rate,avg_as_second_score_rate,avg_max_move_sec,worst_max_move_sec
0,S18_LA11,1.0,10,12,12,12,0.666667,0.75,0.583333,3.666032,6.126297
1,S20_LA11,1.0,11,12,12,12,0.625000,0.75,0.500000,2.292331,5.116467
2,S20_LA12,1.0,11,12,12,12,0.625000,0.75,0.500000,2.395088,5.397794
3,S22_LA11,1.0,12,12,12,12,0.500000,0.75,0.250000,2.577445,5.951308
4,S22_LA12,1.0,12,12,12,12,0.500000,0.75,0.250000,2.441951,5.529481


In [12]:
# ============================================================
# Timing breakdown from raw GameResult objects
# Use this immediately after a run_match() result list, or after focused tests.
# ============================================================

def timing_breakdown_from_results(results):
    rows = []

    for game_i, r in enumerate(results):
        sub_times = []
        ref_times = []

        for m in r.move_log:
            agent = m.get("agent", "")
            sec = float(m.get("sec", 0.0))

            if agent == "SUB":
                sub_times.append(sec)
            elif agent.startswith("LA"):
                ref_times.append(sec)

        rows.append({
            "game": game_i,
            "submission_mark": r.submission_mark,
            "result_for_submission": r.result_for_submission,
            "plies": r.plies,

            "sub_max_sec": max(sub_times) if sub_times else 0.0,
            "sub_avg_sec": sum(sub_times) / len(sub_times) if sub_times else 0.0,
            "sub_slow_moves_1p8": sum(t >= 1.8 for t in sub_times),
            "sub_slow_moves_2p0": sum(t >= 2.0 for t in sub_times),

            "ref_max_sec": max(ref_times) if ref_times else 0.0,
            "ref_avg_sec": sum(ref_times) / len(ref_times) if ref_times else 0.0,
            "ref_slow_moves_1p8": sum(t >= 1.8 for t in ref_times),
            "ref_slow_moves_2p0": sum(t >= 2.0 for t in ref_times),
        })

    return pd.DataFrame(rows)


# Example for the most recent direct `results` object:
if "results" in globals():
    timing_df = timing_breakdown_from_results(results)
    display(timing_df)

    display(timing_df[[
        "sub_max_sec",
        "sub_avg_sec",
        "sub_slow_moves_1p8",
        "sub_slow_moves_2p0",
        "ref_max_sec",
        "ref_avg_sec",
        "ref_slow_moves_1p8",
        "ref_slow_moves_2p0",
    ]].describe())

,game,submission_mark,result_for_submission,plies,sub_max_sec,sub_avg_sec,sub_slow_moves_1p8,sub_slow_moves_2p0,ref_max_sec,ref_avg_sec,ref_slow_moves_1p8,ref_slow_moves_2p0
0,0,1,loss,40,3.111802,0.305196,2,2,10.756784,1.214897,4,2
1,1,1,loss,40,3.695236,0.354806,2,2,4.780951,0.910008,4,2
2,2,1,loss,40,3.372604,0.319316,2,2,4.862015,0.930662,5,3
3,3,1,loss,40,3.651012,0.354073,2,2,5.820763,1.020850,5,4
4,4,1,loss,40,3.562316,0.351556,2,2,5.573856,1.000793,4,4
5,5,2,loss,39,2.957462,0.349079,2,2,4.494076,0.538003,2,2
6,6,2,loss,39,2.800299,0.294315,2,1,4.459178,0.530121,2,2
7,7,2,loss,39,2.876065,0.347049,2,2,4.605097,0.557614,2,2
8,8,2,loss,39,2.975448,0.343360,2,2,4.567447,0.531686,2,1
9,9,2,loss,39,2.962090,0.348868,2,2,4.549814,0.525653,2,1


,sub_max_sec,sub_avg_sec,sub_slow_moves_1p8,sub_slow_moves_2p0,ref_max_sec,ref_avg_sec,ref_slow_moves_1p8,ref_slow_moves_2p0
count,10.000000,10.000000,10.0,10.000000,10.000000,10.000000,10.000000,10.00000
mean,3.196433,0.336762,2.0,1.900000,5.446998,0.776029,3.200000,2.30000
std,0.341228,0.022096,0.0,0.316228,1.923730,0.265026,1.316561,1.05935
min,2.800299,0.294315,2.0,1.000000,4.459178,0.525653,2.000000,1.00000
25%,2.958619,0.325327,2.0,2.000000,4.554222,0.533265,2.000000,2.00000
50%,3.043625,0.347958,2.0,2.000000,4.693024,0.733811,3.000000,2.00000
75%,3.514888,0.350937,2.0,2.000000,5.395896,0.983260,4.000000,2.75000
max,3.695236,0.354806,2.0,2.000000,10.756784,1.214897,5.000000,4.00000


In [13]:
# # ============================================================
# # Focused retest of best configs
# # ============================================================

# # Pick top configs from breakpoint summary.
# TOP_N = 3
# FOCUSED_GAMES_PER_SIDE = 5

# top_labels = bp_df.head(TOP_N)["label"].tolist()

# focused_submission_sweep = [
#     cfg for cfg in SUBMISSION_SWEEP
#     if cfg["label"] in top_labels
# ]

# # Test only around the interesting zone.
# # Edit this after seeing bp_df.
# FOCUSED_REFERENCE_DEPTHS = [5, 7, 9, 11, 13]

# focused_df = run_sweep(
#     focused_submission_sweep,
#     FOCUSED_REFERENCE_DEPTHS,
#     REFERENCE_LA_KNOBS,
#     games_per_side=FOCUSED_GAMES_PER_SIDE,
#     seed=RANDOM_SEED + 999999,
# )

# display(focused_df[[
#     "label",
#     "reference_depth",
#     "games",
#     "wins",
#     "losses",
#     "draws",
#     "score_rate",
#     "as_first_score_rate",
#     "as_second_score_rate",
#     "avg_plies",
#     "max_move_sec",
# ]])

# focused_bp_df = breakpoint_summary(focused_df)
# display(focused_bp_df)

In [14]:
# ============================================================
# Locate slow submission moves
# ============================================================

SLOW_THRESHOLD = 1.8

slow_rows = []

for game_i, r in enumerate(results):
    for m in r.move_log:
        if m.get("agent") == "SUB" and m.get("sec", 0.0) >= SLOW_THRESHOLD:
            slow_rows.append({
                "game": game_i,
                "submission_mark": r.submission_mark,
                "result": r.result_for_submission,
                "ply": m["ply"],
                "mark": m["mark"],
                "move": m["move"],
                "sec": m["sec"],
                "legal": m["legal"],
            })

slow_moves_df = pd.DataFrame(slow_rows)
display(slow_moves_df)

,game,submission_mark,result,ply,mark,move,sec,legal
0,0,1,loss,20,1,5,3.111802,"[0, 1, 2, 4, 5, 6]"
1,0,1,loss,22,1,6,2.510254,"[0, 1, 2, 4, 5, 6]"
2,1,1,loss,20,1,5,3.695236,"[0, 1, 2, 4, 5, 6]"
3,1,1,loss,22,1,6,2.796108,"[0, 1, 2, 4, 5, 6]"
4,2,1,loss,20,1,5,3.372604,"[0, 1, 2, 4, 5, 6]"
5,2,1,loss,22,1,6,2.487108,"[0, 1, 2, 4, 5, 6]"
6,3,1,loss,20,1,5,3.651012,"[0, 1, 2, 4, 5, 6]"
7,3,1,loss,22,1,6,2.862069,"[0, 1, 2, 4, 5, 6]"
8,4,1,loss,20,1,5,3.562316,"[0, 1, 2, 4, 5, 6]"
9,4,1,loss,22,1,6,2.872799,"[0, 1, 2, 4, 5, 6]"


In [15]:
# ============================================================
# Save results
# ============================================================

OUT_DIR = BASE_DIR / "local_kaggle_tuning_results"
OUT_DIR.mkdir(exist_ok=True)

# Save current single matchup if it exists.
if "summary_df" in globals():
    summary_path = OUT_DIR / "latest_match_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved:", summary_path)

if "games_df" in globals():
    games_path = OUT_DIR / "latest_match_games.csv"
    games_df.to_csv(games_path, index=False)
    print("Saved:", games_path)

# Save sweep results if they exist.
if "sweep_df" in globals():
    sweep_path = OUT_DIR / "sweep_results.csv"
    sweep_df.to_csv(sweep_path, index=False)
    print("Saved:", sweep_path)

if "bp_df" in globals():
    bp_path = OUT_DIR / "breakpoint_summary.csv"
    bp_df.to_csv(bp_path, index=False)
    print("Saved:", bp_path)

if "focused_df" in globals():
    focused_path = OUT_DIR / "focused_sweep_results.csv"
    focused_df.to_csv(focused_path, index=False)
    print("Saved:", focused_path)

if "focused_bp_df" in globals():
    focused_bp_path = OUT_DIR / "focused_breakpoint_summary.csv"
    focused_bp_df.to_csv(focused_bp_path, index=False)
    print("Saved:", focused_bp_path)

Saved: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\local_kaggle_tuning_results\latest_match_summary.csv
Saved: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\local_kaggle_tuning_results\latest_match_games.csv
Saved: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\local_kaggle_tuning_results\sweep_results.csv
Saved: C:\Users\Uporabnik\Documents\JS\Connect4\Code\Jupiter\Connect4\Kaggle\submission\local_kaggle_tuning_results\breakpoint_summary.csv
